In [8]:
#Instalamos la libreria de STREAMLIT
%pip install streamlit


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
#Instalamos la libreria de PLOTLY
%pip install plotly


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
#Para instalar npm en visual studio
#1.Desde Google escribir node.js
#2. Instalar la versión más recomendada
! npm install localtunnel

⠙⠹⠸⠼⠴
up to date, audited 23 packages in 617ms
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴
3 high severity vulnerabilities

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠴npm notice
npm notice New major version of npm available! 11.19.0 -> 12.2.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.2.1
npm notice To update run: npm install -g npm@12.2.1
npm notice
⠴

In [13]:
%%writefile app.py 
#Creamos el archivo de la APP en el interprete principal (Phyton)
#####################################################
#Importamos librerias
import streamlit as st
import plotly.express as px
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy import stats
######################################################
#Definimos la instancia
@st.cache_resource
######################################################
#Creamos la función de carga de datos
def load_data():
   #Lectura del archivo csv
   df=pd.read_csv("salidas_almacen_SIN_NULOS.csv")
    #Rellenamos nulos
   df =df.bfill()
   df =df.ffill()  
  
   return df
###############################################################################
#Cargo los datos obtenidos de la función "load_data"
df = load_data()
###############################################################################
#CREACIÓN DEL DASHBOARD
#Generamos las páginas que utilizaremos en el diseño
##############################################################################
#Generamos los encabezados para la barra lateral (sidebar)
st.sidebar.title("Salidas Almacén")

#Widget 1: Selectbox
#Menu desplegable de opciones de laa páginas seleccionadas
View= st.sidebar.selectbox(label= "Tipo de Análisis", options= ["Extracción de Características", 
                                              "Regresión Lineal", "Regresión No Lineal",
                                              "Regresión Logística", "ANOVA"])
  ####################################################################################
 
 # CONTENIDO DE LA VISTA 5
if View == "ANOVA":
    #ANOVA 
    #Generamos la lista de variables numéricas
    numeric_df = df.select_dtypes(['float','int'])  #Devuelve Columnas
    Lista_num= numeric_df.columns                   #Devuelve lista de Columnas numéricas
    Lista_cat= ['fecha','nombre_articulo','marca','nombre_almacen']  #Devuelve lista de Columnas categóricas
    #Select box
    Variable_y= st.sidebar.selectbox(label= "Variable dependiente (y)", options= Lista_num)
    #Multi-select box
    Variable_x= st.sidebar.selectbox(label="Variable independiente (x)", options= Lista_cat)  

    #Generamos los encabezados para el dashboard
    st.title("ANOVA")  
   


    #Generamos el diseño del Layout deseado
    # Fila 1
    Contenedor_A, Contenedor_B = st.columns(2)
    #Redefinimos las variables 
    Var_num= df[Variable_y]
    Var_cat= df[Variable_x]  
    
    # Contenedor A: ANOVA y resultados
    with Contenedor_A: 
        st.subheader("Resultados del ANOVA")
        grupos = [grupo[Variable_y].dropna() for nombre, grupo in df.groupby(Variable_x)]
        f_valor, p_valor = stats.f_oneway(*grupos)

        st.write(f"**F-valor:** {f_valor:.3f}")
        st.write(f"**p-valor:** {p_valor:.4f}")
        if p_valor < 0.05:
            st.success("¡Hay diferencias significativas entre los grupos!")
        else:
            st.warning("No hay diferencias significativas entre los grupos.")

    with Contenedor_B:

        st.subheader("Gráfico de Barras: Media por grupo")

        resumen = (
            df.groupby(Variable_x)[Variable_y]
            .agg(['mean', 'sem'])
            .reset_index()
        )

        if not resumen.empty:

            # Selector de categorías
            num_categorias = st.slider(
                "Número de categorías a visualizar",
                min_value=1,
                max_value=len(resumen),
                value=min(10, len(resumen))
            )

            resumen_grafica = resumen.head(num_categorias)

            # Crear gráfica original
            fig, ax = plt.subplots(figsize=(12, 6))

            x = resumen_grafica[Variable_x].astype(str)
            y = resumen_grafica['mean']
            yerr = resumen_grafica['sem'].fillna(0)

            ax.bar(
                x,
                y,
                yerr=yerr,
                capsize=5,
                color='skyblue',
                edgecolor='black'
            )

            ##########################################
            # ESCALA ESPECIAL SOLO PARA AÑO
            ##########################################

            if Variable_y == "anio":

                minimo = (y - yerr).min()
                maximo = (y + yerr).max()

                if pd.notna(minimo) and pd.notna(maximo):

                    margen = max(
                        (maximo - minimo) * 0.10,
                        1
                    )

                    ax.set_ylim(
                        minimo - margen,
                        maximo + margen
                    )

                # Años sin decimales
                ax.yaxis.set_major_locator(
                    MaxNLocator(integer=True)
                )

            else:

                # Escala normal para todas las
                # demás variables numéricas
                ax.relim()
                ax.autoscale_view()

            ##########################################
            # FORMATO ORIGINAL
            ##########################################

            ax.set_ylabel(f"Media de {Variable_y}")
            ax.set_xlabel(Variable_x)
            ax.set_title("Media por grupo con Error Estándar")

            plt.xticks(
                rotation=45,
                ha='right',
                fontsize=9
            )

            plt.tight_layout()

            st.pyplot(fig)

            plt.close(fig)

        else:
            st.warning(
                "No hay datos disponibles para graficar."
            )


Overwriting app.py
